# 05. Direct Preference Optimization (DPO)

## 학습 목표
- RLHF의 복잡성 문제를 DPO가 어떻게 해결하는지 이해
- DPO의 수학적 유도 과정 따라가기
- DPO Loss를 PyTorch로 직접 구현
- HuggingFace TRL의 DPOTrainer 사용법 익히기

## 핵심 논문
- [Direct Preference Optimization: Your Language Model is Secretly a Reward Model (Rafailov et al., 2023)](https://arxiv.org/abs/2305.18290)

---

In [ ]:
# Google Colab 환경 설정
!pip install -q transformers datasets accelerate peft trl

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. RLHF의 복잡성: Reward Model + PPO = 불안정, 구현 어려움

이전 노트북에서 배운 RLHF의 문제를 정리하면:

### RLHF의 어려운 점들

```
RLHF 학습 시 동시에 관리해야 하는 것:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Policy Model       (학습 대상, gradient 계산)
2. Reference Model    (KL 계산용, 동결)
3. Reward Model       (보상 계산, 동결)
4. Value Model        (PPO의 critic, gradient 계산)
5. PPO 알고리즘       (clip ratio, advantage, GAE, ...)
6. KL coefficient     (너무 크면 학습 안 됨, 너무 작으면 diverge)
7. Reward scaling     (보상 정규화)
```

→ **하이퍼파라미터 튜닝이 매우 어렵고, 학습이 불안정하다.**

### DPO의 핵심 질문

> "Reward Model을 학습하고 그걸로 PPO를 돌리는 것이 정말 필요한가?"
> 
> "인간 선호 데이터에서 **직접** 정책을 최적화할 수 없을까?"

---
## 2. DPO 핵심 아이디어: Reward Model 없이 직접 정책 최적화

DPO는 놀라운 수학적 발견에 기반한다:

> **최적의 reward function은 최적의 policy로부터 closed-form으로 유도할 수 있다.**

따라서:
1. Reward Model을 따로 학습할 필요가 없다
2. PPO를 사용할 필요가 없다
3. **SFT 모델 + 선호 데이터만으로** 직접 alignment 가능

```
RLHF:  선호 데이터 → Reward Model → PPO → Aligned Model
DPO:   선호 데이터 ──────────────────────→ Aligned Model
```

---
## 3. DPO 수식 유도: Bradley-Terry → Optimal Policy → Closed-form Loss

### Step 1: RLHF의 목적 함수 (복습)

$$\max_{\pi_\theta} \; \mathbb{E}_{x \sim D, y \sim \pi_\theta} \left[ r(x, y) \right] - \beta \cdot D_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}})$$

### Step 2: 최적 정책의 Closed-form Solution

위 최적화 문제의 최적해는:

$$\pi^*(y|x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y|x) \exp\left(\frac{r(x,y)}{\beta}\right)$$

여기서 $Z(x) = \sum_y \pi_{\text{ref}}(y|x) \exp(r(x,y)/\beta)$ 는 정규화 상수.

### Step 3: Reward를 Policy로 재표현

위 식을 $r(x,y)$에 대해 풀면:

$$r(x, y) = \beta \log \frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x)$$

→ **Reward는 policy와 reference policy의 log ratio로 표현할 수 있다!**

### Step 4: Bradley-Terry에 대입

$$P(y_w \succ y_l | x) = \sigma(r(x, y_w) - r(x, y_l))$$

여기에 Step 3의 reward를 대입하면 ($Z(x)$가 상쇄됨!):

$$P(y_w \succ y_l | x) = \sigma\left(\beta \log \frac{\pi^*(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi^*(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)$$

### Step 5: DPO Loss (최종)

$$\boxed{\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right) \right]}$$

### 직관

- $\pi_\theta(y_w|x) / \pi_{\text{ref}}(y_w|x)$: chosen 응답의 확률이 reference 대비 **증가**해야 함
- $\pi_\theta(y_l|x) / \pi_{\text{ref}}(y_l|x)$: rejected 응답의 확률이 reference 대비 **감소**해야 함
- 두 ratio의 차이가 클수록 loss가 낮아짐

In [ ]:
# DPO 수식 유도 시각화

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. log ratio의 의미
ax = axes[0]
ratio = np.linspace(0.1, 5, 100)
log_ratio = np.log(ratio)
ax.plot(ratio, log_ratio, 'b-', linewidth=2)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=1, color='gray', linestyle='--', alpha=0.5)
ax.fill_between(ratio, log_ratio, 0, where=(ratio > 1), alpha=0.1, color='green')
ax.fill_between(ratio, log_ratio, 0, where=(ratio < 1), alpha=0.1, color='red')
ax.set_xlabel('pi_theta / pi_ref', fontsize=11)
ax.set_ylabel('log(pi_theta / pi_ref)', fontsize=11)
ax.set_title('Log Ratio (Policy vs Reference)', fontsize=12)
ax.text(3, 0.5, 'policy가\n더 높은 확률', fontsize=9, color='green')
ax.text(0.3, -1, 'policy가\n더 낮은 확률', fontsize=9, color='red')
ax.grid(True, alpha=0.3)

# 2. DPO가 원하는 방향
ax = axes[1]
chosen_ratio = np.linspace(-3, 5, 100)
rejected_ratio = np.linspace(-3, 5, 100)
C, R = np.meshgrid(chosen_ratio, rejected_ratio)
# DPO loss = -log(sigma(beta * (chosen_log_ratio - rejected_log_ratio)))
beta = 0.1
Z = -np.log(1 / (1 + np.exp(-(C - R))))
contour = ax.contourf(C, R, Z, levels=20, cmap='RdYlGn_r')
plt.colorbar(contour, ax=ax, label='DPO Loss')
ax.plot([-3, 5], [-3, 5], 'k--', alpha=0.5, label='chosen = rejected')
ax.set_xlabel('log(pi/pi_ref) for chosen', fontsize=10)
ax.set_ylabel('log(pi/pi_ref) for rejected', fontsize=10)
ax.set_title('DPO Loss Landscape', fontsize=12)
ax.annotate('Goal: here', xy=(3, -1), fontsize=10, fontweight='bold',
            color='white', bbox=dict(boxstyle='round', facecolor='green', alpha=0.8))
ax.legend(fontsize=9)

# 3. Beta에 따른 DPO loss
ax = axes[2]
margin = np.linspace(-5, 5, 200)  # chosen_log_ratio - rejected_log_ratio
for beta_val in [0.05, 0.1, 0.5, 1.0]:
    loss = -np.log(1 / (1 + np.exp(-beta_val * margin)))
    ax.plot(margin, loss, label=f'beta={beta_val}', linewidth=2)
ax.set_xlabel('log_ratio(chosen) - log_ratio(rejected)', fontsize=10)
ax.set_ylabel('DPO Loss', fontsize=10)
ax.set_title('DPO Loss by Beta', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 5)

plt.tight_layout()
plt.show()

---
## 4. DPO Loss 직접 구현: PyTorch로 loss function 작성

In [ ]:
def dpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    reference_chosen_logps: torch.Tensor,
    reference_rejected_logps: torch.Tensor,
    beta: float = 0.1,
) -> torch.Tensor:
    """
    DPO Loss 직접 구현.
    
    L_DPO = -E[log sigma(beta * (log(pi(y_w|x)/pi_ref(y_w|x)) - log(pi(y_l|x)/pi_ref(y_l|x))))]
    
    Args:
        policy_chosen_logps: 현재 정책의 chosen 응답 log 확률
        policy_rejected_logps: 현재 정책의 rejected 응답 log 확률
        reference_chosen_logps: reference 정책의 chosen 응답 log 확률
        reference_rejected_logps: reference 정책의 rejected 응답 log 확률
        beta: KL 제약 강도
    
    Returns:
        DPO loss (scalar)
    """
    # Log ratios
    chosen_log_ratios = policy_chosen_logps - reference_chosen_logps
    rejected_log_ratios = policy_rejected_logps - reference_rejected_logps
    
    # DPO loss
    logits = beta * (chosen_log_ratios - rejected_log_ratios)
    loss = -F.logsigmoid(logits).mean()
    
    return loss


# DPO Loss 동작 확인
print("DPO Loss 동작 확인")
print("=" * 50)

# Case 1: Policy가 chosen을 더 좋아하게 된 경우 (좋은 학습)
loss_good = dpo_loss(
    policy_chosen_logps=torch.tensor([-1.0, -0.5]),      # chosen 확률 높음
    policy_rejected_logps=torch.tensor([-3.0, -2.5]),    # rejected 확률 낮음
    reference_chosen_logps=torch.tensor([-2.0, -1.5]),   # ref는 중간
    reference_rejected_logps=torch.tensor([-2.0, -1.5]), # ref는 중간
    beta=0.1
)
print(f"Case 1 (chosen >> rejected): Loss = {loss_good.item():.4f} (낮아야 좋음)")

# Case 2: Policy가 아직 구분을 못하는 경우
loss_neutral = dpo_loss(
    policy_chosen_logps=torch.tensor([-2.0, -1.5]),
    policy_rejected_logps=torch.tensor([-2.0, -1.5]),
    reference_chosen_logps=torch.tensor([-2.0, -1.5]),
    reference_rejected_logps=torch.tensor([-2.0, -1.5]),
    beta=0.1
)
print(f"Case 2 (chosen == rejected): Loss = {loss_neutral.item():.4f} (log(2) = {np.log(2):.4f})")

# Case 3: Policy가 rejected를 더 좋아하는 경우 (나쁜 상태)
loss_bad = dpo_loss(
    policy_chosen_logps=torch.tensor([-3.0, -2.5]),
    policy_rejected_logps=torch.tensor([-1.0, -0.5]),
    reference_chosen_logps=torch.tensor([-2.0, -1.5]),
    reference_rejected_logps=torch.tensor([-2.0, -1.5]),
    beta=0.1
)
print(f"Case 3 (rejected >> chosen): Loss = {loss_bad.item():.4f} (높은 loss!)")

In [ ]:
# 실제 모델에서 log probability 계산하는 방법

def compute_log_probs(model, input_ids, attention_mask, labels):
    """
    모델의 텍스트에 대한 log probability 계산.
    
    P(sequence) = product of P(token_t | token_1, ..., token_{t-1})
    log P(sequence) = sum of log P(token_t | ...)
    """
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
    
    # Shift: logits[t]는 token[t+1]을 예측
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]
    shift_mask = attention_mask[:, 1:]
    
    # Per-token log probability
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)
    
    # Mask padding + sum
    masked_log_probs = token_log_probs * shift_mask
    sequence_log_probs = masked_log_probs.sum(dim=-1)
    
    return sequence_log_probs

# 예시: GPT-2에서 log probability 계산
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

texts = [
    "The capital of France is Paris.",   # 자연스러운 문장
    "The capital of France is banana.",   # 비정상적인 문장
]

inputs = tokenizer(texts, return_tensors='pt', padding=True).to(device)
log_probs = compute_log_probs(model, inputs['input_ids'], inputs['attention_mask'], inputs['input_ids'])

for text, lp in zip(texts, log_probs):
    print(f"Text: '{text}'")
    print(f"  Log prob: {lp.item():.4f}")
    print(f"  → 확률이 {'높음' if lp > log_probs.mean() else '낮음'} (자연스러운 문장일수록 높음)")
    print()

---
## 5. RLHF vs DPO 비교

| 항목 | RLHF (PPO) | DPO |
|------|-----------|-----|
| **필요한 모델** | Policy + Ref + Reward + Value (4개) | Policy + Ref (2개) |
| **학습 알고리즘** | PPO (복잡한 RL) | 단순 cross-entropy류 loss |
| **안정성** | 불안정 (하이퍼파라미터 민감) | 안정적 |
| **구현 난이도** | 높음 | 낮음 |
| **메모리** | ~84 GB (7B) | ~56 GB (7B) |
| **수학적 동치** | - | RLHF와 동일한 최적해 |
| **Reward Model** | 별도 학습 필요 | 불필요 (implicit) |
| **Online 학습** | 가능 (새 응답 생성) | 오프라인 (고정 데이터) |
| **성능** | 좋음 | 비슷하거나 동등 |
| **실무 채택** | 초기 (ChatGPT 등) | 최근 트렌드 (Zephyr, Tulu 등) |

In [ ]:
# RLHF vs DPO 파이프라인 비교 시각화
import matplotlib.patches as mpatches

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# RLHF 파이프라인
ax = axes[0]
steps_rlhf = [
    ('Preference\nData', '#E3F2FD', 0),
    ('Train\nReward Model', '#FFF3E0', 2.5),
    ('Generate\nResponses', '#F3E5F5', 5),
    ('Score with\nReward Model', '#FFF3E0', 7.5),
    ('PPO\nUpdate', '#FFEBEE', 10),
    ('Aligned\nModel', '#E8F5E9', 12.5),
]
for text, color, x in steps_rlhf:
    rect = mpatches.FancyBboxPatch((x, 0.3), 2, 1.4, boxstyle='round,pad=0.15',
                                    facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + 1, 1, text, ha='center', va='center', fontsize=9, fontweight='bold')

for i in range(len(steps_rlhf) - 1):
    ax.annotate('', xy=(steps_rlhf[i+1][2], 1),
                xytext=(steps_rlhf[i][2] + 2, 1),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

ax.set_xlim(-0.5, 15)
ax.set_ylim(-0.2, 2.2)
ax.set_title('RLHF Pipeline (Complex)', fontsize=13, fontweight='bold', color='red')
ax.axis('off')

# DPO 파이프라인
ax = axes[1]
steps_dpo = [
    ('Preference\nData', '#E3F2FD', 1.5),
    ('DPO Loss\n(Direct)', '#E8F5E9', 6),
    ('Aligned\nModel', '#E8F5E9', 10.5),
]
for text, color, x in steps_dpo:
    rect = mpatches.FancyBboxPatch((x, 0.3), 2.5, 1.4, boxstyle='round,pad=0.15',
                                    facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + 1.25, 1, text, ha='center', va='center', fontsize=10, fontweight='bold')

for i in range(len(steps_dpo) - 1):
    ax.annotate('', xy=(steps_dpo[i+1][2], 1),
                xytext=(steps_dpo[i][2] + 2.5, 1),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

ax.set_xlim(-0.5, 15)
ax.set_ylim(-0.2, 2.2)
ax.set_title('DPO Pipeline (Simple)', fontsize=13, fontweight='bold', color='green')
ax.axis('off')

plt.tight_layout()
plt.show()

print("DPO는 Reward Model 학습과 PPO 단계를 건너뛰고,")
print("선호 데이터에서 직접 정책을 최적화한다.")

---
## 6. 실습: HuggingFace TRL DPOTrainer 사용

TRL (Transformer Reinforcement Learning) 라이브러리의 `DPOTrainer`를 사용하여 실제 DPO 학습을 수행한다.

In [ ]:
# Step 1: 선호 데이터 준비 (chosen vs rejected)
# DPOTrainer는 prompt, chosen, rejected 형식의 데이터를 기대

preference_data = [
    {
        "prompt": "What is machine learning?",
        "chosen": "Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data and make predictions without being explicitly programmed for each task.",
        "rejected": "Machine learning is computers."
    },
    {
        "prompt": "Explain what a neural network is.",
        "chosen": "A neural network is a computing system inspired by biological neural networks. It consists of layers of interconnected nodes (neurons) that process information and learn to recognize patterns.",
        "rejected": "A neural network is a type of network that is neural."
    },
    {
        "prompt": "What is overfitting?",
        "chosen": "Overfitting occurs when a model learns the training data too well, including noise and outliers, resulting in excellent training performance but poor generalization to new unseen data.",
        "rejected": "Overfitting is bad for models."
    },
    {
        "prompt": "What is gradient descent?",
        "chosen": "Gradient descent is an optimization algorithm that iteratively adjusts model parameters by moving in the direction of steepest decrease of the loss function, helping the model converge to optimal weights.",
        "rejected": "Gradient descent is when you go down."
    },
    {
        "prompt": "What is transfer learning?",
        "chosen": "Transfer learning is a technique where knowledge gained from training on one task is applied to a different but related task, allowing models to leverage pre-existing knowledge and require less data.",
        "rejected": "Transfer learning transfers things."
    },
    {
        "prompt": "What is attention mechanism?",
        "chosen": "The attention mechanism allows a model to dynamically focus on different parts of the input sequence when producing each element of the output, weighing the importance of each input token.",
        "rejected": "Attention is paying attention to stuff."
    },
    {
        "prompt": "Explain regularization.",
        "chosen": "Regularization is a set of techniques used to prevent overfitting by adding constraints or penalties to the model, such as L1/L2 weight penalties, dropout, or early stopping.",
        "rejected": "Regularization makes things regular."
    },
    {
        "prompt": "What is backpropagation?",
        "chosen": "Backpropagation is an algorithm for training neural networks that computes the gradient of the loss function with respect to each weight by applying the chain rule from the output layer back to the input.",
        "rejected": "Backpropagation goes backward."
    },
]

# 데이터 증폭
preference_data = preference_data * 20  # 160개
dpo_dataset = Dataset.from_list(preference_data)

print(f"DPO 학습 데이터: {len(dpo_dataset)} 개")
print(f"\n예시:")
print(f"  Prompt:   {dpo_dataset[0]['prompt']}")
print(f"  Chosen:   {dpo_dataset[0]['chosen'][:80]}...")
print(f"  Rejected: {dpo_dataset[0]['rejected']}")

In [ ]:
# Step 2: 모델 준비
from trl import DPOTrainer, DPOConfig
from transformers import TrainingArguments

model_name = 'gpt2'

# Policy model (학습 대상)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Reference model (동결, KL 계산용)
ref_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Model: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Step 3: DPOTrainer 설정 + 학습
dpo_config = DPOConfig(
    output_dir='./gpt2-dpo',
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    beta=0.1,                # DPO의 beta 파라미터
    max_length=256,
    max_prompt_length=128,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_strategy='no',
    report_to='none',
    remove_unused_columns=False,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

print("DPO 학습 시작...")
train_result = dpo_trainer.train()
print(f"\n학습 완료! Final loss: {train_result.training_loss:.4f}")

In [ ]:
# Step 4: 학습 결과 확인

# Loss curve
log_history = dpo_trainer.state.log_history
steps = [e['step'] for e in log_history if 'loss' in e]
losses = [e['loss'] for e in log_history if 'loss' in e]

if steps:
    plt.figure(figsize=(10, 4))
    plt.plot(steps, losses, 'g-', alpha=0.7, linewidth=2)
    plt.axhline(y=np.log(2), color='gray', linestyle='--', alpha=0.5, label=f'Random baseline (ln2={np.log(2):.3f})')
    plt.xlabel('Step')
    plt.ylabel('DPO Loss')
    plt.title('DPO Training Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"초기 Loss: {losses[0]:.4f} (random: {np.log(2):.4f})")
    print(f"최종 Loss: {losses[-1]:.4f}")
    print(f"→ Loss가 ln(2) 아래로 내려가면 모델이 선호를 학습하고 있는 것!")

In [ ]:
# Step 5: DPO 학습 후 생성 테스트
model.eval()

test_prompts = [
    "What is deep learning?",
    "Explain what LoRA is.",
    "What is reinforcement learning?",
]

print("DPO-aligned 모델 생성 결과:")
print("=" * 60)

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=80, do_sample=True,
            temperature=0.7, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nQ: {prompt}")
    print(f"A: {response[len(prompt):].strip()[:200]}")
    print("-" * 60)

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: Beta 하이퍼파라미터 실험

DPO의 beta 값을 {0.01, 0.1, 0.5}로 바꿔가며 학습하고, 각각의 loss curve를 비교하세요.
- beta가 작을수록 loss가 어떻게 변하는지 관찰
- beta가 클수록 모델의 생성 결과가 어떻게 달라지는지 비교

In [ ]:
# TODO: Beta 실험
# Hint:
# betas = [0.01, 0.1, 0.5]
# results = {}
# for beta in betas:
#     1. 새 model 로드
#     2. DPOConfig(beta=beta, ...) 설정
#     3. DPOTrainer로 학습
#     4. loss curve 기록
#     5. 동일 프롬프트로 생성 테스트
# 마지막에 beta별 loss curve를 하나의 그래프에 그리기

---
## 핵심 정리

| 개념 | 설명 | 핵심 포인트 |
|------|------|-------------|
| RLHF 복잡성 | 4개 모델 + PPO + 많은 하이퍼파라미터 | 구현과 튜닝이 어려움 |
| DPO 핵심 아이디어 | Reward Model 없이 직접 최적화 | 수학적으로 RLHF와 동일한 최적해 |
| DPO 수식 | $-\log\sigma(\beta(\log\frac{\pi}{\pi_{ref}}\text{(chosen)} - \log\frac{\pi}{\pi_{ref}}\text{(rejected)}))$ | chosen 확률 증가, rejected 확률 감소 |
| Beta ($\beta$) | KL 제약 강도 | 크면 보수적, 작으면 공격적 |
| DPOTrainer | TRL 라이브러리의 간편한 API | prompt + chosen + rejected 데이터만 필요 |
| 실무 트렌드 | DPO가 RLHF를 빠르게 대체 중 | Zephyr, Tulu, Mixtral 등 |

**다음 노트북**: [06-evaluation.ipynb](06-evaluation.ipynb) - LLM 평가 방법론